In [46]:
pip install selenium

In [1]:
pip install webdriver-manager

In [37]:
from selenium import webdriver
from selenium.webdriver.common.by import By #https://www.youtube.com/watch?v=Dt66BsSlIgM https://www.youtube.com/watch?v=Dt66BsSlIgM
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
from time import sleep

In [5]:
def GetEmployer(url0):
    page0 = requests.get(url0)
    soup0 = BeautifulSoup(page0.text, 'html')
    title = soup0.find_all('div', {'class': 'company__name line-clamp-2'})[0].text.strip() if soup0.find_all('div', {'class': 'company__name line-clamp-2'}) else None
    review = soup0.find_all('div', {'class': 'company__indicator-number'})[0].text.strip() if soup0.find_all('div', {'class': 'company__indicator-number'}) else None
    rec_share = soup0.find_all('div', {'class': 'company__indicator-number'})[1].text.strip() if soup0.find_all('div', {'class': 'company__indicator-number'}) else None
    review_cnts = soup0.find_all('span', {'class': 'tabs__count'})[0].text.strip().replace("\xa0", "") if soup0.find_all('span', {'class': 'tabs__count'}) else None
    review_name = soup0.find_all('div', {'class': 'dashboard-sidebar__grade-humanly'})[0].text.strip() if soup0.find_all('div', {'class': 'dashboard-sidebar__grade-humanly'}) else None
    page1 = requests.get(url0+'/career')
    soup1 = BeautifulSoup(page1.text, 'html')
    about = soup1.find('div', {'class': 'row career__about-items'})
    data = {}
    for item in about.find_all('div', {'class':'career__about-item'}):
        key = item.find('div').text.strip() if item.find('div') else None
        value = item.find('p').text.strip() if item.find('p') else None
        data[key] = value
    emp_cnt = data.get('Количество сотрудников')
    city = data.get('Город (головной офис)')
    year_founded = data.get('Год основания')
    industry = data.get('Отрасль')
    site = soup1.find('div', {'class': 'col-md-6 col-lg-4 career__about-item'}).find('span', {'onclick': "$(this).closest('form').submit()"}).text.strip() if soup1.find('div', {'class': 'col-md-6 col-lg-4 career__about-item'}).find('span', {'onclick': "$(this).closest('form').submit()"}) else None
    descr = soup1.find('div', {'class': 'text-16 line-clamp-4'}).text.strip().replace("\xa0", "").replace("\n", "") if soup1.find('div', {'class': 'text-16 line-clamp-4'}) else None
    return title, review, rec_share, review_cnts, review_name, emp_cnt, city, year_founded, industry, site, descr

In [47]:
import sys
import os

!apt-get purge google-chrome-stable chromium-browser chromium-chromedriver -y #https://www.geeksforgeeks.org/python/how-to-do-web-scraping-using-selenium-and-google-colab/
!apt-get autoremove -y
!apt-get autoclean -y
!apt-get update
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb #https://medium.com/@arjuns0206/running-selenium-on-google-colab-a118d10ca5f8
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -y --fix-broken
!apt-get install -y google-chrome-stable
!pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service #https://www.selenium.dev/documentation/webdriver/
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.binary_location = '/usr/bin/google-chrome'
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
print("Можем начинать, всё подгрузила для selenium")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
The following packages were automatically installed and are no longer required:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  google-chrome-stable*
0 upgraded, 0 newly installed, 1 to remove and 21 not upgraded.
After this operation, 414 MB disk space will be freed.
(Reading database ... 118623 files and directories currently installed.)
Removing google-chrome-stable (147.0.7727.55-1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
(Reading database ... 118343 files and directories currently installed.)
P

In [48]:
urls = set()

In [49]:
try:
  url = 'https://dreamjob.ru/categories'
  driver.get(url)
  sleep(2)
  while True:
    try:
      page = driver.page_source
      soup = BeautifulSoup(page, 'html')
      for link in soup.find_all('a'):
        if link.get('href'): # and link.get('href').startswith('https://'):
            link = re.findall(r'/employers/\d+', link.get('href'))
            if (len(link) > 0):
              urls.add(link[0])
      show_more_btn = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "show_more_news"))) #https://selenium-python.readthedocs.io/waits.html
      if show_more_btn.is_displayed():
        driver.execute_script("arguments[0].scrollIntoView();", show_more_btn)
        sleep(1)
        show_more_btn.click()
        print("Кнопку нажали, ждемс дальше")
        sleep(2)
      else:
        print("кнопочки нет")
        break
    except TimeoutException:
      print("Кнопка больше не найдена, все ссылки надеюсь загружены")
      break
    except NoSuchElementException:
      print("Кнопка не найдена, всё загрузили")
      break

  urls = list(urls)
  print(urls)
finally:
  driver.quit()

Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопку нажали, ждемс дальше
Кнопка больше не найдена, все ссылки надеюсь загружены
['/employers/37960', '/employers/56617', '/employers/25875', '/employers/60461', '/employers/102120', '/employers/25996', '/employers/82786', '/employers/89675', '/employers/41678', '/employers/287864', '/employers/104885', '/employers/104814', '/employers/32153', '/employers/56708', '/employers/44340', '/employers/46854', '/employers/2672030', '/employers/45131', '/employers/25928', '/employers/2590161', '/employers/39409', '/employers/51073', '/employers/55900', '/employers/48785', '/employers/44845', '/employers/25894', '/employers/26118', '/employers/26842', '/employers/36507', '/employers/55484', '/employers/35549', '/employers/104914', '/employers/573

In [50]:
len(urls)

397

In [51]:
employers = []

In [52]:
for link in urls:
    print(link)
    res = GetEmployer('https://dreamjob.ru/'+link)
    employers.append(res)

/employers/37960
/employers/56617
/employers/25875
/employers/60461
/employers/102120
/employers/25996
/employers/82786
/employers/89675
/employers/41678
/employers/287864
/employers/104885
/employers/104814
/employers/32153
/employers/56708
/employers/44340
/employers/46854
/employers/2672030
/employers/45131
/employers/25928
/employers/2590161
/employers/39409
/employers/51073
/employers/55900
/employers/48785
/employers/44845
/employers/25894
/employers/26118
/employers/26842
/employers/36507
/employers/55484
/employers/35549
/employers/104914
/employers/57362
/employers/53815
/employers/25609
/employers/57356
/employers/48941
/employers/25466
/employers/88890
/employers/25951
/employers/26984
/employers/56818
/employers/91247
/employers/56706
/employers/172969
/employers/61577
/employers/26820
/employers/37657
/employers/106108
/employers/26155
/employers/102469
/employers/25953
/employers/92373
/employers/93692
/employers/104895
/employers/26102
/employers/38623
/employers/25926
/

In [53]:
import pandas as pd
df = pd.DataFrame(employers)
df.columns = ['title', 'review', 'rec_share', 'review_cnts', 'review_name', 'emp_cnt', 'city', 'year_founded', 'industry', 'site', 'descr']
df.head()

,title,review,rec_share,review_cnts,review_name,emp_cnt,city,year_founded,industry,site,descr
0,ЛУКОЙЛ-Центрнефтепродукт,"3,7",72%,544,Хорошо,1000-9999,Москва,None,Нефть и газ,http://www.luknef.lukoil.ru,None
1,FES retail,"4,0",76%,816,Очень хорошо,1000-9999,Москва,None,Розничная торговля,None,None
2,Полиметалл,"4,1",83%,511,Очень хорошо,более 10000,Санкт-Петербург,None,Добывающая отрасль,http://www.polymetal.ru,None
3,САТУРН - строительные материалы,"4,1",78%,494,Очень хорошо,1000-9999,Санкт-Петербург,None,Розничная торговля,http://www.saturn.net,None
4,МОНЭКС ТРЕЙДИНГ,"3,9",76%,1033,Хорошо,1000-9999,Москва,1994,Розничная торговля,http://www.moneks.ru,None


In [54]:
df.to_excel('dreamjob.xlsx', index=False)